In [0]:
# --------------------------------------------------------------
# Imports
# --------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

# --------------------------------------------------------------
# Spark session (in Databricks a SparkSession already exists)
# --------------------------------------------------------------
spark = SparkSession.builder.getOrCreate()

In [0]:
# --------------------------------------------------------------
# 1. Load the bronze_events Delta table
# --------------------------------------------------------------
# DBFS mounts (/mnt/...) are disabled in this workspace — load from Unity Catalog instead
bronze_df = spark.read.table("workspace.default.bronze_events")

# Quick sanity check
display(bronze_df.select(
    F.count("*").alias("total_rows"),
    F.approx_count_distinct("employee_id").alias("num_employees")
).limit(1))

total_rows,num_employees
162337,150


In [0]:
# --------------------------------------------------------------
# 2. Derive columns that make the later aggregation easier
# --------------------------------------------------------------
events = (
    bronze_df
    .withColumn("date", F.to_date("timestamp"))                     # daily grain
    .withColumn("hour", F.hour("timestamp"))                       # hour of day (0‑23)

    # flag for “after‑hours” activity (outside 09:00‑18:00)
    .withColumn(
        "is_after_hours",
        F.when( (F.col("hour") < 9) | (F.col("hour") > 18), F.lit(1) ).otherwise(F.lit(0))
    )
    # flag for external email (only keep rows that are really external)
    .withColumn(
        "is_external_email",
        F.when( (F.col("activity") == "email_external") & (F.col("email_external") == "external"), 1 )
         .otherwise(0)
    )
    # flag for file‑open events (so we can count files separately)
    .withColumn(
        "is_file",
        F.when(F.col("activity") == "file_open", 1).otherwise(0)
    )
    # flag for confidential file‑open (only relevant when activity = file_open)
    .withColumn(
        "is_confidential_file",
        F.when( (F.col("activity") == "file_open") & (F.col("is_confidential") == True), 1)
         .otherwise(0)
    )
    # flag for USB‑connect events
    .withColumn(
        "is_usb",
        F.when(F.col("activity") == "usb_connect", 1).otherwise(0)
    )
)

events.printSchema()

root
 |-- employee_id: long (nullable = true)
 |-- department: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- activity: string (nullable = true)
 |-- file_size_bytes: long (nullable = true)
 |-- email_external: string (nullable = true)
 |-- usb_device_id: string (nullable = true)
 |-- is_confidential: boolean (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- is_after_hours: integer (nullable = false)
 |-- is_external_email: integer (nullable = false)
 |-- is_file: integer (nullable = false)
 |-- is_confidential_file: integer (nullable = false)
 |-- is_usb: integer (nullable = false)



In [0]:
# --------------------------------------------------------------
# 3. Aggregate to one row per employee per day
# --------------------------------------------------------------
daily_agg = (
    events
    .groupBy("employee_id", "date")
    .agg(
        F.sum("is_file").alias("files_cnt"),
        F.sum("is_confidential_file").alias("confidential_files_cnt"),
        F.sum("is_usb").alias("usb_cnt"),
        F.sum("is_external_email").alias("external_emails_cnt"),
        F.sum("is_after_hours").alias("after_hours_cnt")
    )
    .orderBy("employee_id", "date")
)

display(daily_agg.limit(5))

employee_id,date,files_cnt,confidential_files_cnt,usb_cnt,external_emails_cnt,after_hours_cnt
1,2023-10-01,9,0,8,7,19
1,2023-10-02,5,0,10,3,19
1,2023-10-03,10,0,10,4,22
1,2023-10-04,7,1,14,6,20
1,2023-10-05,9,0,12,6,21


In [0]:
# --------------------------------------------------------------
# 4. Window that spans all days of a given employee
# --------------------------------------------------------------
emp_window = Window.partitionBy("employee_id")

# Helper to build a (value, mean, std, z) trio for a column
def add_z(col_name: str):
    return (
        F.col(col_name),
        F.avg(col_name).over(emp_window).alias(f"{col_name}_mean"),
        F.stddev(col_name).over(emp_window).alias(f"{col_name}_std")
    )

# Apply the helper for each of the five count columns
daily_z = daily_agg.select(
    "employee_id",
    "date",
    *add_z("files_cnt"),
    *add_z("confidential_files_cnt"),
    *add_z("usb_cnt"),
    *add_z("external_emails_cnt"),
    *add_z("after_hours_cnt")
)

# Now compute the Z‑score.  If stddev = 0 (no variation) we set z = 0.
z_expr = lambda cnt, mean, std: (
    F.when(std == 0, F.lit(0.0))
     .otherwise( (cnt - mean) / std )
)

gold_features = daily_z.select(
    "employee_id",
    "date",
    "files_cnt",
    "confidential_files_cnt",
    "usb_cnt",
    "external_emails_cnt",
    "after_hours_cnt",

    # Z‑scores
    z_expr(F.col("files_cnt"),          F.col("files_cnt_mean"),          F.col("files_cnt_std")).alias("files_z"),
    z_expr(F.col("confidential_files_cnt"), F.col("confidential_files_cnt_mean"), F.col("confidential_files_cnt_std")).alias("confidential_files_z"),
    z_expr(F.col("usb_cnt"),            F.col("usb_cnt_mean"),            F.col("usb_cnt_std")).alias("usb_z"),
    z_expr(F.col("external_emails_cnt"),F.col("external_emails_cnt_mean"),F.col("external_emails_cnt_std")).alias("external_emails_z"),
    z_expr(F.col("after_hours_cnt"),   F.col("after_hours_cnt_mean"),   F.col("after_hours_cnt_std")).alias("after_hours_z")
)

display(gold_features.limit(5))

employee_id,date,files_cnt,confidential_files_cnt,usb_cnt,external_emails_cnt,after_hours_cnt,files_z,confidential_files_z,usb_z,external_emails_z,after_hours_z
1,2023-10-01,9,0,8,7,19,0.13988620785159123,-0.7912518229508043,-0.2776734596426306,0.4514909099143749,-1.067075586239545
1,2023-10-02,5,0,10,3,19,-1.5387482863675004,-0.7912518229508043,0.5156792821934576,-1.2416000022645315,-1.067075586239545
1,2023-10-03,10,0,10,4,22,0.5595448314063641,-0.7912518229508043,0.5156792821934576,-0.8183272742198048,0.9336911379596
1,2023-10-04,7,1,14,6,20,-0.6994310392579546,0.15825036459016084,2.1023847658656343,0.028218181869648338,-0.40015334483982995
1,2023-10-05,9,0,12,6,21,0.13988620785159123,-0.7912518229508043,1.3090320240295459,0.028218181869648338,0.26676889655988506


In [0]:
# --------------------------------------------------------------
# 5. Persist the feature set
# --------------------------------------------------------------
# DBFS mounts (/mnt/...) are disabled in this workspace — persist as a
# Unity Catalog managed Delta table instead
gold_features.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_features")

print("✅ Gold table 'workspace.default.gold_features' created successfully.")

✅ Gold table 'workspace.default.gold_features' created successfully.
